In [1]:
!pip install datasets transformers -q

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from datasets import load_dataset


print("Loading CoDet-M4 dataset...")

dataset = load_dataset(
    "DaniilOr/CoDET-M4",
    trust_remote_code=True
)

print(dataset)
print("\nColumn names:", dataset['train'].column_names)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'DaniilOr/CoDET-M4' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading CoDet-M4 dataset...


README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

dataset_without_comments.parquet:   0%|          | 0.00/458M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500552 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__'],
        num_rows: 500552
    })
})

Column names: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__']


In [ ]:
train_df = dataset['train'].to_pandas()

print("Full dataset size:", len(train_df))
print("Columns:", train_df.columns.tolist())
print("\nSplit column values:", train_df['split'].value_counts())
print("\nTarget column values:", train_df['target'].value_counts())
print("\nLanguage column values:", train_df['language'].value_counts())
train_df.head(2)

Full dataset size: 500552
Columns: ['code', 'language', 'model', 'split', 'target', 'source', 'features', 'cleaned_code', '__index_level_0__']

Split column values: split
train    405069
test      47749
val       47734
Name: count, dtype: int64

Target column values: target
ai       254331
human    246221
Name: count, dtype: int64

Language column values: language
python    185163
java      174169
cpp       141220
Name: count, dtype: int64


,code,language,model,split,target,source,features,cleaned_code,__index_level_0__
0,"def order_phase_diagram(lines, stable_entries,...",python,human,train,human,gh,"{'avgFunctionLength': 1.0, 'avgIdentifierLengt...","def order_phase_diagram(lines, stable_entries,...",20280
1,"def reload(self):\n """"""Reload catalog i...",python,human,train,human,gh,"{'avgFunctionLength': 1.0, 'avgIdentifierLengt...",def reload(self):\n \n if time.t...,45860


In [5]:
train_py = train_df[
    train_df['split'] == 'train'
].reset_index(drop=True)

val_py = train_df[
    train_df['split'] == 'val'
].reset_index(drop=True)

test_py = train_df[
    train_df['split'] == 'test'
].reset_index(drop=True)


print(f"Train size: {len(train_py)}")
print(f"Val size:   {len(val_py)}")
print(f"Test size:  {len(test_py)}")

print("\nLabel distribution (train):")
print(train_py['target'].value_counts())

print("\nLanguage distribution (train):")
print(train_py['language'].value_counts())

Train size: 405069
Val size:   47734
Test size:  47749

Label distribution (train):
target
ai       207982
human    197087
Name: count, dtype: int64

Language distribution (train):
language
python    149763
java      140875
cpp       114431
Name: count, dtype: int64


In [ ]:
import re
import pandas as pd

def preprocess_code(code):

    if not isinstance(code, str):
        return ""
    code = code.replace("\r\n", "\n")
    code = "\n".join([line.rstrip() for line in code.split("\n")])
    code = re.sub(r"\n\s*\n+", "\n\n", code)

    return code.strip()

In [ ]:
def clean_dataframe(df):

    print("Original size:", len(df))

    df = df.dropna(subset=["cleaned_code"])
    df = df.dropna(subset=["target"])
    df = df[df["target"].isin(["human", "ai"])]
    df = df[df["cleaned_code"].str.strip() != ""]
    df = df.drop_duplicates(subset=["cleaned_code"])
    df = df[df["cleaned_code"].str.len() > 10]

    print("After cleaning:", len(df))
    return df.reset_index(drop=True)

In [8]:
train_py = clean_dataframe(train_py)
val_py   = clean_dataframe(val_py)
test_py  = clean_dataframe(test_py)

train_py["code"] = train_py["cleaned_code"].apply(preprocess_code)
val_py["code"]   = val_py["cleaned_code"].apply(preprocess_code)
test_py["code"]  = test_py["cleaned_code"].apply(preprocess_code)

print("Preprocessing complete.")

Original size: 405069
After cleaning: 395805
Original size: 47734
After cleaning: 46956
Original size: 47749
After cleaning: 47132
Preprocessing complete.


In [9]:
label_map = {
    "human": 0,
    "ai": 1
}

train_py["label"] = train_py["target"].map(label_map)
val_py["label"]   = val_py["target"].map(label_map)
test_py["label"]  = test_py["target"].map(label_map)

print("Label conversion complete.")

Label conversion complete.


In [10]:
train_py.to_csv("train_python.csv", index=False)
val_py.to_csv("val_python.csv", index=False)
test_py.to_csv("test_python.csv", index=False)

print("Saved successfully:")
print("train_python.csv")
print("val_python.csv")
print("test_python.csv")

Saved successfully:
train_python.csv
val_python.csv
test_python.csv


In [11]:
train_df = pd.read_csv("train_python.csv")
val_df   = pd.read_csv("val_python.csv")
test_df  = pd.read_csv("test_python.csv")

print(len(train_df), len(val_df), len(test_df))

395805 46956 47132


In [14]:
X_train = train_df["code"]
y_train = train_df["label"]

X_val = val_df["code"]
y_val = val_df["label"]

X_test = test_df["code"]
y_test = test_df["label"]

In [15]:
C_values = [0.01, 0.1, 1, 5, 10]

best_f1 = 0
best_C = None

for C in C_values:

    model = Pipeline([

        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3,5),
            min_df=2,
            max_features=80000
        )),

        ("svm", LinearSVC(C=C, random_state=42))

    ])

    model.fit(X_train, y_train)

    preds = model.predict(X_val)

    f1 = f1_score(y_val, preds)

    print(f"C={C}  →  Val F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_C = C


print("\nBest C:", best_C)

C=0.01  →  Val F1=0.9070
C=0.1  →  Val F1=0.9400
C=1  →  Val F1=0.9509
C=5  →  Val F1=0.9504
C=10  →  Val F1=0.9488

Best C: 1


In [16]:
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])


final_model = Pipeline([

    ("tfidf", TfidfVectorizer(
        analyzer="char",
        ngram_range=(3,5),
        min_df=2,
        max_features=50000
    )),

    ("svm", LinearSVC(
        C=best_C,
        random_state=42
    ))

])


final_model.fit(X_train_full, y_train_full)

print("Final model trained on train + validation")

Final model trained on train + validation


In [17]:
test_preds = final_model.predict(X_test)

test_metrics = {

    "accuracy": accuracy_score(y_test, test_preds),
    "precision": precision_score(y_test, test_preds),
    "recall": recall_score(y_test, test_preds),
    "f1": f1_score(y_test, test_preds)

}

test_metrics

{'accuracy': 0.9554655011457184,
 'precision': 0.9573026696973749,
 'recall': 0.9495579133510168,
 'f1': 0.9534145637747742}

In [18]:
print(classification_report(
    y_test,
    test_preds,
    target_names=["Human", "AI"]
))

              precision    recall  f1-score   support

       Human       0.95      0.96      0.96     24512
          AI       0.96      0.95      0.95     22620

    accuracy                           0.96     47132
   macro avg       0.96      0.96      0.96     47132
weighted avg       0.96      0.96      0.96     47132



In [19]:
pd.DataFrame([test_metrics]).to_csv(
    "baseline1_tfidf_svm_results.csv",
    index=False
)

print("Saved results to baseline1_tfidf_svm_results.csv")

Saved results to baseline1_tfidf_svm_results.csv
